# 6 Thomson-Reuters Tick History intraday data


In [1]:
import pandas as pd
import numpy as np

import re
import os

import pdb

### Parallelization

In [2]:
import dask
dask.config.set(scheduler="processes")

#@dask.delayed
def load_TRTH_trade(filename,
             tz_exchange="America/New_York",
             only_non_special_trades=True,
             only_regular_trading_hours=True,
             open_time="09:30:00",
             close_time="16:00:00",
             merge_sub_trades=True):
    try:
        if re.search('(csv|csv\\.gz)$',filename):
            DF = pd.read_csv(filename)
        if re.search(r'arrow$',filename):
            DF = pd.read_arrow(filename)
        if re.search('parquet$',filename):
            DF = pd.read_parquet(filename)

    except Exception as e:
     #   print("load_TRTH_trade could not load "+filename)
     #   print(e)
        return None
    
    try:
        DF.shape
    except Exception as e: # DF does not exist
        print("DF does not exist")
        print(e)
        return None

    
    if DF.shape[0]==0:
        return None
    
    if only_non_special_trades:
        DF = DF[DF["trade-stringflag"]=="uncategorized"]

    DF.drop(columns=["trade-rawflag","trade-stringflag"],axis=1,inplace=True)
    
    DF.index = pd.to_datetime(DF["xltime"],unit="d",origin="1899-12-30",utc=True)
    DF.index = DF.index.tz_convert(tz_exchange)  # .P stands for Arca, which is based at New York
    DF.drop(columns="xltime",inplace=True)
    
    if only_regular_trading_hours:
        DF=DF.between_time(open_time,close_time)    # warning: ever heard e.g. about Thanksgivings?
    
    if merge_sub_trades:
           DF=DF.groupby(DF.index).agg(trade_price=pd.NamedAgg(column='trade-price', aggfunc='mean'),
                                       trade_volume=pd.NamedAgg(column='trade-volume', aggfunc='sum'))
    
    return DF



#@dask.delayed
def load_TRTH_bbo(filename,
             tz_exchange="America/New_York",
             only_regular_trading_hours=True,
             merge_sub_trades=True):
    try:
        if re.search(r'(csv|csv\.gz)$',filename):
            DF = pd.read_csv(filename)
        if re.search(r'arrow$',filename):
            DF = pd.read_arrow(filename)
        if re.search(r'parquet$',filename):
            DF = pd.read_parquet(filename) 
    except Exception as e:
       # print("load_TRTH_bbo could not load "+filename)
        return None
    
    try:
        DF.shape
    except Exception as e: # DF does not exist
        print("DF does not exist")
        print(e)
        return None

    if DF.shape[0]==0:
        return None
    
        
    DF.index = pd.to_datetime(DF["xltime"],unit="d",origin="1899-12-30",utc=True)
    DF.index = DF.index.tz_convert(tz_exchange)  # .P stands for Arca, which is based at New York
    DF.drop(columns="xltime",inplace=True)
    
    if only_regular_trading_hours:
        DF=DF.between_time("09:30:00","16:00:00")    # ever heard about Thanksgivings?
        
    if merge_sub_trades:
        DF=DF.groupby(DF.index).last()
    

        
    return DF

In [3]:
@dask.delayed
def load_merge_trade_bbo(ticker,date,
                         country="US",
                         dirBase="data/raw/TRTH/equities/",
                         suffix="parquet",
                         suffix_save=None,
                         dirSaveBase="data/clean/TRTH/equities/events",
                         saveOnly=False,
                         doSave=False
                        ):
    
    file_trade=dirBase+"/"+country+"/trade/"+ticker+"/"+str(date.date())+"-"+ticker+"-trade."+suffix
    file_bbo=file_trade.replace("trade","bbo")
    trades=load_TRTH_trade(file_trade)
    bbos  =load_TRTH_bbo(file_bbo)
    try:
        trades.shape + bbos.shape
    except:
        return None
    
    events=trades.join(bbos,how="outer")
    
    if doSave:
        dirSave=dirSaveBase+"/"+country+"/events/"+ticker
        if not os.path.isdir(dirSave):
            os.makedirs(dirSave)

        if suffix_save:
            suffix=suffix_save
        
        file_events=dirSave+"/"+str(date.date())+"-"+ticker+"-events"+"."+suffix
       # pdb.set_trace()

        saved=False
        if suffix=="arrow":
            events.to_arrow(file_events)
            saved=True
        if suffix=="parquet":
         #   pdb.set_trace()
            events.to_parquet(file_events,use_deprecated_int96_timestamps=True)
            saved=True
            
        if not saved:
            print("suffix "+suffix+" : format not recognized")
            
        if saveOnly:
            return saved
    return events

In [4]:
from datetime import datetime

ticker="SPY.P"

startDate="2010-01-01"
endDate="2010-12-31"

datelist = pd.date_range(startDate,endDate).tolist()

In [5]:
%time allpromises=[load_merge_trade_bbo("SPY.P",date,saveOnly=True,doSave=True,suffix="parquet",suffix_save="parquet") for date in datelist]

CPU times: total: 62.5 ms
Wall time: 52 ms


Thus, it takes almost no time at all to create execution promises. Let us check that we really have promises:

In [6]:
allpromises[0]

Delayed('load_merge_trade_bbo-f1130690-11a3-418e-ab11-46832e857635')

To actually perform a computation, simply call the compute() function

In [7]:
%%time
allpromises[0].compute()

CPU times: total: 46.9 ms
Wall time: 2.27 s


Now, let us load all the files in a parallel way

In [8]:
%%time
alldata=dask.compute(allpromises) 

CPU times: total: 938 ms
Wall time: 4.29 s


#### Delayed, other ways


There are alternative ways to delay a function: use dask.delayed(some_function) directly. 

In [9]:
allpromises=[dask.delayed(pd.read_csv)(fn) for fn in allfiles]

NameError: name 'allfiles' is not defined

or defined a delayed version of a function

In [10]:
load_TRTH_trade_delayed=dask.delayed(load_TRTH_trade)

In [11]:
del alldata  # cleanup




 

### Merge trades and bbo data

If one wishes to create a single dataframe, then one can proceeed in the following way. 

In [12]:
import glob

trade_files=glob.glob("data/raw/TRTH/equity/US/trade/SPY.P/2009*")
trade_files.sort()

allpromises=[load_TRTH_trade(fn) for fn in trade_files]
trades=dask.compute(allpromises)[0]

trades=pd.concat(trades)

In [13]:
bbo_files=glob.glob("data/raw/TRTH/equity/US/bbo/SPY.P/2009*")
bbo_files.sort()

allpromises=[load_TRTH_bbo(fn) for fn in bbo_files]
bbos=dask.compute(allpromises)[0]

bbos=pd.concat(bbos)

In [14]:
%time events=trades.join(bbos,how="outer")    # keep everything

CPU times: total: 32.5 s
Wall time: 35.2 s


In [15]:
events.shape

(73068257, 6)

We are entering into the realms of big data. Let us save this object

In [17]:
# before saving a parquet object, we need to ensure that the columns are in numeric format
events["bid-price"]=events["bid-price"].values.astype("float")
events["bid-volume"]=events["bid-volume"].values.astype("float")
events["ask-price"]=events["ask-price"].values.astype("float")
events["ask-volume"]=events["ask-volume"].values.astype("float")

#so far, one still needs to add the use_deprectated_int96_timestamps option
events.to_parquet("SPY_2009_events.parquet")

## join_asof

If one is interested in the state of the LOB before a trade takes place, one can use `pd.merge_asof(trades,bbos)`means exactly that 


In [18]:
%time events=pd.merge_asof(trades,bbos,left_index=True,right_index=True)

CPU times: total: 7.42 s
Wall time: 9.29 s


In [19]:
events.shape

(10451615, 6)

This was equivalent to join with how="outer", forward fill, and then remove lines without data for trade_price and trade_volume. So, much faster and much more memory efficient.